In [10]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -qU langchain langchain-community langchain-core langchain-huggingface chromadb sentence-transformers
!pip install -qU transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 경로 설정
LOCAL_MODEL_PATH = "/content/drive/MyDrive/DILAB/Models/Qwen2-7B-Instruct"

# 벡터 DB가 저장된 경로
BASE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment"
DB_PATH = os.path.join(BASE_DIR, "VectorDB")

# 2. 벡터 DB 로드
print(f"[백터 DB 경로]: {DB_PATH}")

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print("벡터 DB 로드 완료")

# 3. 로컬 Qwen 모델 로드
print(f"[로컬 모델 경로]: {LOCAL_MODEL_PATH}")

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"{LOCAL_MODEL_PATH}")
else:
    try:
        # 1. 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)

        # 2. 양자화 설정 (4bit 로딩을 위한 설정 객체 생성)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        # 3. 모델 로드
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL_PATH,
            device_map="auto",
            quantization_config=bnb_config,
            dtype=torch.float16
        )

        # 4. 파이프라인 구축
        text_generator = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=512,
            temperature=0.1
        )
        print("Qwen2-7B-Instruct 모델 로드 완료")

    except Exception as e:
        print(e)

[백터 DB 경로]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터 DB 로드 완료
[로컬 모델 경로]: /content/drive/MyDrive/DILAB/Models/Qwen2-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2-7B-Instruct 모델 로드 완료


In [15]:
import warnings
from transformers import logging

# 0. 경고 메시지 차단
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

def experiment_rag_comparison(question):
    print(f"\n{'='*80}")
    print(f"[질문]: {question}")
    print(f"{'='*80}\n")

    # [Case 1] 순수 Qwen (RAG 미적용)
    print("[1. RAG 적용 전 (Original Qwen)]")

    messages_no_rag = [
        {"role": "system", "content": "당신은 e스포츠 전문가입니다. 질문에 답변해 주세요."},
        {"role": "user", "content": question}
    ]
    prompt_no_rag = tokenizer.apply_chat_template(messages_no_rag, tokenize=False, add_generation_prompt=True)

    output_no_rag = text_generator(
        prompt_no_rag,
        max_new_tokens=512,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_no_rag[0]['generated_text'].strip()}")
    print(f"\n{'-'*80}\n")

    # [Case 2] RAG 적용
    print("[2. RAG 적용 후]")

    raw_docs = vector_db.similarity_search(question, k=10) # 넉넉히 10개 가져옴
    docs = []
    seen_titles = set()

    for doc in raw_docs:
        title = doc.metadata.get("title", "제목 없음")
        if title in seen_titles:
            continue
        seen_titles.add(title)
        docs.append(doc)
        if len(docs) >= 5:
            break

    context_text = ""
    print(f"\n[검색된 근거 자료]")
    if not docs:
        print("검색된 문서 없음")
    else:
        for i, doc in enumerate(docs):
            title = doc.metadata.get("title", "제목 없음")
            date = doc.metadata.get("date", "날짜 미상")
            preview = doc.page_content.replace("\n", " ").strip()[:80]
            print(f"      [{i+1}] {title} ({date}) -> {preview}...")
            context_text += f"문서{i+1}: {doc.page_content}\n\n"

    # 2. 프롬프트 구성
    system_prompt = f"""
    당신은 사실만 전달하는 냉철한 e스포츠 분석가입니다.
    오직 아래 제공된 [참고 자료]만을 근거로 답변해야 합니다.

    [주의사항 - 필독]
    1. **시점 구분 필수:** 기사 내용이 '경기 전 예상(Preview)'인지 '경기 후 결과(Review)'인지 엄격하게 구분하세요.
       - "승리할 것으로 보인다", "우세가 점쳐진다"는 결과가 아닙니다.
       - 반드시 "승리했다", "제압했다", "2:0으로 이겼다"와 같은 **확정된 과거형 문장**만 결과로 인정하세요.
    2. **날짜 대조:** 질문에 특정 날짜가 있다면, 기사 작성일이나 기사 내의 날짜 정보를 확인하여 정확히 매칭되는지 확인하세요.
    3. **환각 방지:** [참고 자료]에 없는 내용은 절대, 절대로 지어내지 마세요. 모르면 솔직하게 "제공된 뉴스 기사에는 해당 경기 결과 정보가 없습니다"라고 답하세요.
    4. **답변 형식:** 질문에 대한 핵심(승패, 스코어, 주요 선수)만 간결하게 요약해서 답변하세요. 사족은 뺍니다.

    [참고 자료]
    {context_text}
    """

    messages_rag = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompt_rag = tokenizer.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)

    # 3. 실제 입력된 프롬프트 출력
    print(f"\n[LLM에게 실제로 들어가는 입력 데이터 (Prompt)]")
    print(f"   {prompt_rag[:300]} \n   ... (중략: 기사 본문들) ... \n   {prompt_rag[-200:]}")

    # 4. 답변 생성
    output_rag = text_generator(
        prompt_rag,
        max_new_tokens=1024,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_rag[0]['generated_text'].strip()}")
    print(f"\n{'='*80}")

# 실행
experiment_rag_comparison("2026년 1월 23일의 경기 결과를 알려줘")


[질문]: 2026년 1월 23일의 경기 결과를 알려줘

[1. RAG 적용 전 (Original Qwen)]

답변:
죄송합니다, 하지만 현재 시점에서 2026년 1월 23일의 경기 결과를 알 수 없습니다. 스포츠 경기의 결과는 미래에 대한 정보로, 그 정보는 해당 날짜가 실제로 도래한 후에만 알 수 있습니다. 따라서 이 질문에 대한 정확한 답변을 제공하는 것은 불가능합니다.

--------------------------------------------------------------------------------

[2. RAG 적용 후]

[검색된 근거 자료]
      [1] 2026 LCK컵 2주 차…10승씩 나눠 가진 바론 그룹과 장로 그룹 (2026-01-26) -> 제목: 2026 LCK컵 2주 차…10승씩 나눠 가진 바론 그룹과 장로 그룹...
      [2] 2026 LCK컵 판도 바꿀 '슈퍼 위크' 열린다 (2026-01-27) -> 내용: 10개 팀이 바론-장로 그룹으로 나뉘어 진행되고 있는 2026 LCK컵에서 두 그룹이 나란히 10승씩을 따내며 접전을 펼치고 있다. 20...
      [3] 박준석 감독 '다음 경기 승리 다짐하며'[포토] (2026-01-22) -> 제목: 박준석 감독 '다음 경기 승리 다짐하며'[포토] 내용: (엑스포츠뉴스 종로, 박지영 기자) 22일 오후 서울 청진동 롤파크에서 열린 '2...
      [4] 2026 LCK컵, '체급이 다르다' 증명한 DK... 브리온 54분 셧아웃, 장로 그룹 2연승 질주 (2026-01-14) -> 제목: 2026 LCK컵, '체급이 다르다' 증명한 DK... 브리온 54분 셧아웃, 장로 그룹 2연승 질주...
      [5] T1 vs KT, 다시 만난다…LCK컵서 '롤드컵 결승 리매치' 성사 (2026-01-20) -> 내용: 2주 차 빅매치 예고…슈퍼 위크 앞두고 선두권 판도 가를 승부 (MHN 양진희 기자) '2025 롤드컵 결승전'에서 명승부를 펼쳤던 

In [ ]:
!pip install -qU langchain langchain-community langchain-core rank_bm25 sentence-transformers chromadb

In [ ]:
import warnings
from transformers import logging
import numpy as np
import os

# 1. 경고 메시지 끄기
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# 2. 필수 라이브러리 임포트
try:
    from langchain_community.retrievers import BM25Retriever
    from sentence_transformers import CrossEncoder
    from langchain_core.documents import Document
except ImportError:
    os.system("pip install -qU langchain-community rank_bm25 sentence-transformers chromadb")
    from langchain_community.retrievers import BM25Retriever
    from sentence_transformers import CrossEncoder
    from langchain_core.documents import Document

# 3. 리랭커 모델 로드
if 'reranker_model' not in globals():
    reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3', automodel_args={'torch_dtype': 'auto'})
else:
    print("리랭커 모델이 이미 로드되어 있습니다.")

리랭커 모델이 이미 로드되어 있습니다.


In [ ]:
# [앙상블 검색 + 리랭킹 함수]
import numpy as np

def advanced_search(question, k_final=5):
    k_candidate = 30

    # (A) 벡터 검색
    vector_docs = vector_db.similarity_search(question, k=k_candidate)

    # (B) BM25 검색
    all_data = vector_db.get()
    bm25_docs = [Document(page_content=t, metadata=m)
                 for t, m in zip(all_data['documents'], all_data['metadatas'])]

    bm25_retriever = BM25Retriever.from_documents(bm25_docs)
    bm25_retriever.k = k_candidate
    keyword_docs = bm25_retriever.invoke(question)

    # (C) 합치기 (중복 제거)
    combined_docs = vector_docs + keyword_docs
    unique_docs = {}
    for doc in combined_docs:
        unique_docs[doc.page_content] = doc

    candidate_docs = list(unique_docs.values())

    if not candidate_docs:
        return []

    # 2단계: 리랭킹
    pairs = [[question, doc.page_content] for doc in candidate_docs]
    scores = reranker_model.predict(pairs) # 여기서 -10 ~ 10 사이 점수가 나옴

    # 경기 결과 데이터(Match_Result) 가산점 부여
    final_scores = []
    for i, doc in enumerate(candidate_docs):
        original_score = scores[i]
        source = doc.metadata.get('source', '')

        # 만약 크롤링한 경기 결과 데이터라면?
        if source == 'Match_Result':
            boosted_score = original_score + 100.0
            # print(f"[가산점 적용] {doc.metadata.get('title')} : {original_score:.4f} -> {boosted_score:.4f}")
            final_scores.append(boosted_score)
        else:
            final_scores.append(original_score)

    # 3단계: 정렬 및 상위 k개 추출
    sorted_indices = np.argsort(final_scores)[::-1]

    final_docs = []
    print(f"\n[통합 검색 및 리랭킹 결과 상위 {k_final}개]")

    for i in range(min(k_final, len(candidate_docs))):
        idx = sorted_indices[i]
        doc = candidate_docs[idx]
        score = final_scores[idx]

        # 가산점 안 받은 애들 중 점수가 너무 낮으면 컷
        if score < -5: continue

        final_docs.append(doc)

        # 디버깅 출력 (출처 확인용)
        title = doc.metadata.get('title', '제목없음')
        source = doc.metadata.get('source', 'Unknown')
        print(f"   [{i+1}등] 점수: {score:.4f} | [출처: {source}] {title}")

    return final_docs

# 2. 최종 질문 함수 (프롬프트 강화)
def ask_smart_rag(question):
    print(f"\n{'='*80}")
    print(f"[RAG 질문]: {question}")

    relevant_docs = advanced_search(question, k_final=5)

    if not relevant_docs:
        print("관련된 문서를 찾지 못했습니다.")
        return

    # 문맥 만들기
    context_text = ""
    for i, doc in enumerate(relevant_docs):
        src = doc.metadata.get('source', 'Unknown')
        context_text += f"문서{i+1} [출처: {src}]: {doc.page_content}\n\n"

    # 프롬프트: 사실 우선 원칙 주입
    system_prompt = f"""
    당신은 팩트 기반의 e스포츠 분석가입니다.
    제공된 [문서]를 바탕으로 질문에 정확하게 답하세요.

    [중요 규칙 - 서열 정리]
    1. **우선순위:** `[출처: Match_Result]`는 공식 기록(Fact)입니다. 뉴스 기사(`News_Article`)보다 내용을 우선시하세요.
    2. **내용 충돌:** 만약 뉴스 기사와 경기 결과(Match_Result)의 내용(승패, 스코어)이 다르다면, 무조건 `Match_Result`를 정답으로 채택하세요.
    3. **정확성:** 경기 결과나 스코어를 말할 때는 `Match_Result`에 적힌 그대로만 말하세요.
    4. **보완:** 뉴스 기사는 경기의 분위기나 선수 인터뷰 등 부가적인 설명을 할 때만 사용하세요.

    [문서]
    {context_text}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 답변 생성
    output = text_generator(prompt, max_new_tokens=1024, max_length=None, return_full_text=False)
    print(f"\n[답변]:\n{output[0]['generated_text'].strip()}")
    print(f"{'='*80}")


In [ ]:

ask_smart_rag("2026년 LCK CUP에서 DRX의 경기 결과를 알려줘")


[RAG 질문]: 2026년 LCK CUP에서 DRX의 경기 결과를 알려줘


ValueError: not enough values to unpack (expected 3, got 0)

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 임베딩 모델 로드 (Octen-8B)
embedding_model = HuggingFaceEmbeddings(
    model_name="Octen/Octen-Embedding-8B",
    model_kwargs={
        "device": "cuda",
        "trust_remote_code": True,
        "model_kwargs": {"torch_dtype": torch.float16}
    },
    encode_kwargs={"normalize_embeddings": True}
)

# 2. 새로운 DB 생성 (split_docs는 이전에 생성해둔 청킹 데이터)
# split_docs가 메모리에 남아있어야 해!
print("🚀 새로운 인덱스 생성 중... (A100이라 금방 끝날 거야)")
vector_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=DB_PATH
)

# 저장 확인
print(f"✅ 새 Chroma DB 연결 완료 (문서 개수: {vector_db._collection.count()}개)")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

🚀 새로운 인덱스 생성 중... (A100이라 금방 끝날 거야)


NameError: name 'split_docs' is not defined

In [12]:
import warnings
import torch
from transformers import logging

# 0. 경고 메시지 차단
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# ==============================================================================
# [필독] 아까 세팅에서 꼬인 임베딩 모델을 Octen-8B로 강제 교체 (차원 4096 맞추기)
# ==============================================================================
print("🔄 [System] 검색용 임베딩 모델을 Octen-8B로 교체 중 (A100 전용)...")

# 세팅 코드에서 'cpu'로 잡았던 걸 'cuda'와 'Octen'으로 덮어씌움
embedding_model = HuggingFaceEmbeddings(
    model_name="Octen/Octen-Embedding-8B",
    model_kwargs={
        "device": "cuda",
        "trust_remote_code": True,
        "model_kwargs": {"torch_dtype": torch.float16} # A100용 반정밀도
    },
    encode_kwargs={"normalize_embeddings": True}
)

# DB 연결 재설정 (임베딩 함수가 똑같아야 검색이 됨)
vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print(f"✅ [System] DB 연결 갱신 완료 (문서 개수: {vector_db._collection.count()}개)")


# ==============================================================================
# 1. 실험 함수 정의
# ==============================================================================
def experiment_rag_comparison(question):
    print(f"\n{'='*80}")
    print(f"[질문]: {question}")
    print(f"{'='*80}\n")

    # [Case 1] 순수 Qwen (RAG 미적용)
    print("[1. RAG 적용 전 (Original Qwen)]")

    messages_no_rag = [
        {"role": "system", "content": "당신은 e스포츠 전문가입니다. 질문에 답변해 주세요."},
        {"role": "user", "content": question}
    ]
    prompt_no_rag = tokenizer.apply_chat_template(messages_no_rag, tokenize=False, add_generation_prompt=True)

    output_no_rag = text_generator(
        prompt_no_rag,
        max_new_tokens=512,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_no_rag[0]['generated_text'].strip()}")
    print(f"\n{'-'*80}\n")

    # [Case 2] RAG 적용
    print("[2. RAG 적용 후]")

    # Octen-8B로 4096차원 검색 수행
    raw_docs = vector_db.similarity_search(question, k=10)
    docs = []
    seen_titles = set()

    for doc in raw_docs:
        title = doc.metadata.get("title", "제목 없음")
        if title in seen_titles:
            continue
        seen_titles.add(title)
        docs.append(doc)
        if len(docs) >= 5:
            break

    context_text = ""
    print(f"\n[검색된 근거 자료]")
    if not docs:
        print("❌ 검색된 문서 없음 - DB 경로가 맞는지 확인 필요!")
    else:
        for i, doc in enumerate(docs):
            title = doc.metadata.get("title", "제목 없음")
            date = doc.metadata.get("date", "날짜 미상")
            preview = doc.page_content.replace("\n", " ").strip()[:80]
            print(f"      [{i+1}] {title} ({date}) -> {preview}...")
            context_text += f"문서{i+1}: {doc.page_content}\n\n"

    # 2. 프롬프트 구성 (경기 결과 기사 구분을 위해 강화됨)
    system_prompt = f"""
    당신은 사실만 전달하는 냉철한 e스포츠 분석가입니다.
    오직 아래 제공된 [참고 자료]만을 근거로 답변해야 합니다.

    [주의사항 - 필독]
    1. **시점 구분 필수:** 기사 내용이 '경기 전 예상(Preview)'인지 '경기 후 결과(Review)'인지 엄격하게 구분하세요.
       - "~할 것으로 보인다", "~예상된다"는 결과가 아닙니다.
       - 반드시 "승리했다", "완파했다", "2:0으로 이겼다"와 같은 **과거형 문장**만 경기 결과로 인정하세요.
    2. **날짜 대조:** 질문의 날짜와 기사의 날짜가 일치하는지 반드시 확인하세요.
    3. **환각 방지:** [참고 자료]에 경기 결과가 명확히 없다면 지어내지 말고 "제공된 기사에는 해당 경기의 최종 결과가 없습니다"라고 답하세요.
    4. **답변 형식:** 핵심(승패, 스코어) 위주로 간결하게 답변하세요.

    [참고 자료]
    {context_text}
    """

    messages_rag = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    prompt_rag = tokenizer.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)

    # 3. 실제 입력된 프롬프트 출력
    print(f"\n[LLM 입력 프롬프트 미리보기]")
    print(f"   {prompt_rag[:200]} ... (생략) ... {prompt_rag[-150:]}")

    # 4. 답변 생성
    output_rag = text_generator(
        prompt_rag,
        max_new_tokens=1024,
        max_length=None,
        return_full_text=False
    )
    print(f"\n답변:\n{output_rag[0]['generated_text'].strip()}")
    print(f"\n{'='*80}")

# 실행
experiment_rag_comparison("2026년 LCK CUP에서 펜타킬을 한 선수를 알려줘")

🔄 [System] 검색용 임베딩 모델을 Octen-8B로 교체 중 (A100 전용)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ [System] DB 연결 갱신 완료 (문서 개수: 561개)

[질문]: 2026년 LCK CUP에서 펜타킬을 한 선수를 알려줘

[1. RAG 적용 전 (Original Qwen)]

답변:
죄송합니다, 하지만 현재 시점에서 2026년 LCK CUP의 상세 정보는 알 수 없습니다. 또한, 특정 게임 이벤트나 플레이어의 성과는 미래에 대한 예측이 불가능하며, 실제로 이벤트나 경기의 결과는 여러 요인에 따라 달라질 수 있습니다. 따라서 2026년 LCK CUP에서 펜타킬을 한 선수에 대한 정보는 현재로서는 제공할 수 없습니다. 

그러나 펜타킬이라는 기록을 세우는 것은 매우 드물고 높은 기술과 운이 결합된 결과로, 이는 대부분의 경우 상당한 기량을 가진 선수가 그룹이나 팀에서 이루어집니다. 펜타킬을 이루려면 한 경기에 5명의 적을 모두 죽이는 것이 필요하며, 이는 일반적으로 게임의 끝에 가까운 시점에서 이루어지며, 팀의 전략적 승리와 개인의 기량이 모두 합쳐져야 합니다.

만약 특정 시점이나 상황에서 이 정보를 원하신다면, 해당 정보가 공개된 후에 다시 질문해 주시면 감사하겠습니다.

--------------------------------------------------------------------------------

[2. RAG 적용 후]

[검색된 근거 자료]
      [1] [LCK컵] '에이밍' 시즌 첫 펜타킬! KT, 브리온전 역전승 (2026-01-25) -> 내용: 25일 종로 치지직 롤파크에서 열린 '2026 LCK컵' 그룹 배틀 2주 5일 차 2경기, 브리온과 kt 롤스터의 대결에서 kt 롤스터가...
      [2] [LCK컵] ‘에이밍 시즌 1호 펜타킬’…KT, 브리온 꺾고 연패 탈출 (2026-01-26) -> 제목: [LCK컵] ‘에이밍 시즌 1호 펜타킬’…KT, 브리온 꺾고 연패 탈출 내용: [OSEN=종로, 고용준 기자] 점점 궁지에 몰려 패색이 ...
      [3] 2026 LCK컵 판도 바꿀 '슈퍼 위크' 